In [ ]:
import os
import pandas as pd
from datetime import datetime, date
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

In [ ]:
# connect database
import os
if os.path.exists('pass.env'):
    load_dotenv('pass.env')
else:
    load_dotenv('../pass.env')


db_user = os.getenv('DB_USER')
db_pass = os.getenv('DB_PASS')
db_host = os.getenv('DB_HOST')
db_port = os.getenv('DB_PORT')
db_name = os.getenv('DB_NAME')

db_url = f"postgresql://{db_user}:{db_pass}@{db_host}:{db_port}/{db_name}"
engine = create_engine(db_url)

In [ ]:
# fetch existing data dim_date
with engine.connect() as connection:
    query = "SELECT * FROM dim_date;"
    dim_date = pd.read_sql(text(query), connection)

In [ ]:
dim_date['full_date'] = pd.to_datetime(dim_date['full_date'])

In [ ]:
# define function to identify events
def get_ecommerce_event(dt):
    month = dt.month
    day = dt.day
    weekday = dt.weekday() # 0 is Monday, 6 is Sunday
    
    # Valentine's Day (Brazil - Dia dos Namorados)
    if month == 6 and day == 12:
        return "Dia dos Namorados"
        
    # Consumer Day (Major retail event in Brazil)
    if month == 3 and day == 15:
        return "Consumer Day"
        
    # Mother's Day: 2nd Sunday of May (Days 8-14)
    if month == 5 and weekday == 6 and 8 <= day <= 14:
        return "Mother's Day"
        
    # Father's Day: 2nd Sunday of August (Days 8-14)
    if month == 8 and weekday == 6 and 8 <= day <= 14:
        return "Father's Day"
        
    # Black Friday: Last Friday of November (Days 23-29)
    if month == 11 and weekday == 4 and 23 <= day <= 29:
        return "Black Friday"
        
    # Christmas
    if month == 12 and day == 25:
        return "Christmas"
        
    return None

# Apply event logic
dim_date['event_name'] = dim_date['full_date'].apply(get_ecommerce_event)
dim_date['is_major_sale_event'] = dim_date['event_name'].notnull().astype(int)

In [ ]:
# define payday logic
def is_payday(dt):
    day = dt.day
    if day == 5 or day == 20:
        return 1
    return 0

dim_date['is_payday'] = dim_date['full_date'].apply(is_payday)

In [ ]:
# Writing upgraded dim_date to database
dim_date.to_sql('dim_date', engine, if_exists='replace', index=False)

with engine.connect() as connection:
    connection.execute(text("ALTER TABLE dim_date ADD PRIMARY KEY (full_date);"))
    connection.commit()